# 05 — Listed companies

**The question:** *What did Petrobras earn last quarter — and how do I read a
Brazilian ITR without double-counting the quarter?*

Brazilian quarterly filings have a trap in them that is entirely avoidable and
entirely silent if you miss it: **an ITR publishes the same account twice under
one reference date** — once for the three months, once year-to-date. They are
distinguished only by the period span. Add them and you have counted the quarter
twice.

Endpoints: `company_financials`, `financials`, `lookup`.

In [ ]:
# The SDK is not on PyPI. From the repository root:
#
#     pip install -e sdk/
#
# Auth is the shared publishable key printed in the docs. It is for TESTING:
# everyone reading the docs has the same one, so it identifies the project and
# not you. It puts you on the ANONYMOUS tier. Set SILO_TOKEN to a GitHub
# sign-in token (notebook 00) to run signed in.
import os

os.environ.setdefault("SILO_URL", "https://zcjbtpxuhdekpwcxmepn.supabase.co")
os.environ.setdefault(
    "SILO_ANON_KEY", "sb_publishable__yfFQsykAglrvc9GS6_PYw_B24ex437"
)

import pandas as pd

from silo_client import SiloClient

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

silo = SiloClient()
print(f"tier            : {silo.tier}")
print(f"catalog version : {silo.catalog()['version']}")

In [ ]:
def as_of(*datasets: str) -> pd.DataFrame:
    """Print how fresh every dataset this notebook relies on actually is.

    Run this FIRST, every time. A stale warehouse then shows up in the output
    instead of being silently baked into a number further down.

      as_of            the newest period that has landed AND has elapsed.
                       This is freshness.
      complete_through the newest period classified COMPLETE — what the
                       default windows serve.
      newest_period    the newest period KEY. It can sit in the FUTURE when a
                       family files forward-dated (FIP is keyed 31-December).
                       Never read this as freshness.
      landed_at        when ingest last SUCCEEDED. A later failed run never
                       advances it.
      notes            a caveat the dates cannot carry. Printed in full below,
                       never summarised, never dropped.
    """
    cov = pd.DataFrame(silo.coverage())
    rows = cov[cov["dataset"].isin(datasets)].copy()
    missing = set(datasets) - set(rows["dataset"])
    if missing:
        raise RuntimeError(f"coverage() has no row for {sorted(missing)}")
    print(
        rows[
            ["dataset", "as_of", "complete_through", "newest_period", "landed_at"]
        ].to_string(index=False)
    )
    for r in rows.itertuples():
        if r.notes:
            print(f"\n  CAVEAT [{r.dataset}]\n  {r.notes}")
    return rows.set_index("dataset")

In [ ]:
COVERAGE = as_of("financials")

Note that `complete_through` is `None` for `financials`. Company filings do not
have an industry-wide "complete period" the way monthly fund filings do — each
company files on its own calendar. `as_of` is the newest reference date that has
landed.

## One id, three spellings

A ticker, a CNPJ and a CVM code resolve to the same company — **through CVM's
published FCA *valores mobiliários* map only**, and active listings only. The
CNPJ and the trading code arrive on the same filed row; nothing is matched by
name, and a delisted code resolves to nothing rather than to a guess.

In [ ]:
for ident in ("PETR4", "33000167000101", "9512"):
    rows = silo.company_financials(ident, start="2026-04-01")
    head = rows[0] if rows else {}
    print(f"{ident:>16s} -> id_type={head.get('id_type'):<8s} "
          f"cnpj={head.get('cnpj')} ticker={head.get('ticker')} "
          f"{str(head.get('company'))[:34]}")

In [ ]:
# lookup resolves the other direction. A COMPANY row carries a `tickers` array
# straight off the FCA map — a published mapping, never a name match.
for row in silo.lookup("33000167000101"):
    print(row)

That `tickers` array is what makes "the id you price with is the id you read
fundamentals with" true. Note `PETR*` sitting beside `PETR3` and `PETR4`: the
map is served **as published**, wildcards included.

A name search returns whatever matched, by the key that matched, and does not
invent an edge between the kinds:

In [ ]:
hits = pd.DataFrame(silo.lookup("PETROBRAS"))
print(hits["asset_class"].value_counts().to_frame("rows").to_string())
print()
cia = hits[hits["asset_class"] == "cia"]
print("company rows returned by the NAME search:")
print(cia[["id", "id_type", "name", "cnpj", "tickers"]].to_string(index=False))
print()
print("tickers is None on a company with no ACTIVE published listing. That")
print("means 'not currently listed', not 'we could not find it' — and it is")
print("never filled in from the name.")

## `company_financials` — the headline shape

One row per **(document, reference date, period span)**: revenue, gross profit,
net income, total assets, equity, net margin and ROE.

Default scope is **consolidated** (`con`). Pass `scope="ind"` for individual.

In [ ]:
cf = pd.DataFrame(silo.company_financials("PETR4", start="2024-01-01"))
cf["ref_date"] = pd.to_datetime(cf["ref_date"])
cf = cf.sort_values(["ref_date", "period_months"])

cf[["doc_type", "ref_date", "period_start", "period_end", "period_months",
    "revenue", "net_income", "equity", "net_margin_pct", "roe_pct", "version"]]

## Read `period_months` before comparing two rows

Look at the rows sharing a `ref_date`. Same date, same company, same accounts —
**different spans**.

In [ ]:
by_date = cf.groupby("ref_date")["period_months"].apply(list)
multi = by_date[by_date.map(len) > 1]

print("reference dates published under more than one period span:\n")
for ref, spans in multi.items():
    print(f"  {ref:%Y-%m-%d}  spans: {spans}")
print()
if len(multi):
    ref = multi.index[-1]
    same_date = cf[cf["ref_date"] == ref]
    print(f"{ref:%Y-%m-%d} in full:\n")
    print(same_date[["doc_type", "period_start", "period_end", "period_months",
                     "revenue", "net_income"]]
          .to_string(index=False, float_format=lambda v: f"{v:,.0f}"))

In [ ]:
if len(multi):
    q = same_date[same_date["period_months"] == same_date["period_months"].min()]
    ytd = same_date[same_date["period_months"] == same_date["period_months"].max()]
    if len(q) and len(ytd) and q["period_months"].iloc[0] != ytd["period_months"].iloc[0]:
        wrong = q["net_income"].iloc[0] + ytd["net_income"].iloc[0]
        print("THE TRAP, made concrete:")
        print(f"  {q['period_months'].iloc[0]}-month net income   : "
              f"R$ {q['net_income'].iloc[0]:>18,.0f}")
        print(f"  {ytd['period_months'].iloc[0]}-month net income   : "
              f"R$ {ytd['net_income'].iloc[0]:>18,.0f}")
        print(f"  naive sum                : R$ {wrong:>18,.0f}   <- DOUBLE COUNTS")
        print()
        print("  The quarter is already inside the year-to-date figure. Pick a")
        print("  span and stay on it; never add across period_months.")

### Working on one span

The safe pattern is to filter to a single `period_months` before doing anything
time-series shaped. Quarterly comparisons want the 3-month rows; annual ones
want the 12-month rows from the DFP.

In [ ]:
quarterly = cf[cf["period_months"] == 3].set_index("ref_date")
print(f"{len(quarterly)} quarterly (3-month) observations\n")
print(quarterly[["revenue", "net_income", "net_margin_pct", "roe_pct"]]
      .to_string(float_format=lambda v: f"{v:,.2f}"))
print()
print("CAVEAT: roe_pct is the PERIOD's return on equity and is NOT annualised.")
print("A 3-month row divides one quarter's profit by equity. Multiply it")
print("yourself if you want an annual figure — and say that you did.")

## Restatements: `version`

When a company re-files, only the **newest version** of each statement is
served, and `version` carries it. In `company_financials`, a balance sheet filed
under a different version than the income statement reads `NULL` rather than
being paired across filings — a mismatched pair is not silently assembled into a
ratio.

In [ ]:
print(cf.groupby(["doc_type", "version"]).size().to_frame("rows").to_string())
print()
missing_bs = cf[cf["total_assets"].isna() | cf["equity"].isna()]
print(f"rows where the balance sheet is NULL: {len(missing_bs)}")
if len(missing_bs):
    print(missing_bs[["doc_type", "ref_date", "period_months", "version",
                      "net_income", "total_assets", "equity"]]
          .to_string(index=False))
    print()
    print("NULL here means 'not paired across versions', not 'no balance sheet'.")

## `financials` — the line items, exactly as filed

`company_financials` is a convenience shape. `financials` returns **one row per
account line**, as the company filed it. Nothing is summed, annualised or
restated.

In [ ]:
dre = pd.DataFrame(silo.financials("PETR4", statement="DRE",
                                   start="2026-04-01", end="2026-06-30"))
print(f"{len(dre)} account lines\n")
dre.head(15)

In [ ]:
if not dre.empty:
    spans = sorted(dre["period_months"].dropna().unique())
    print(f"period spans present in this ONE statement: {spans}")
    print()
    print("The same account code appears once per span. Filter first:")
    three = dre[dre["period_months"] == min(spans)]
    print(f"  {min(spans)}-month rows: {len(three)}")
    print(f"  all rows            : {len(dre)}")

### `financials` refuses a window over one page

Like every other set-returning function, it raises `22023` rather than hand back
a slice. Unlike `panel` / `quote_history` / `fund_nav`, it has **no cursor** —
narrow the window, or pin a statement.

In [ ]:
from silo_client.client import SiloOverCap

try:
    silo.financials("PETR4", start="2010-01-01")
except SiloOverCap as exc:
    print("refused:")
    print(" ", exc.body)
    print()
    print("No p_after here. Narrow p_from/p_to, or pass statement= to pin one.")

## Where this goes next

* Notebook `03` goes the other way: which funds hold this company's shares and
  its debentures.
* Notebook `01` prices the same ticker — the id you price with is the id you
  read fundamentals with, which is the point of the FCA map.

---

## The rules this notebook obeyed

* **Nothing was filled.** No forward-fill, no interpolation, no carried-forward
  last observation. A gap in a chart is a gap in the filings.
* **Every caveat was printed beside its number** — `coverage().notes`,
  `catalog().regime_breaks`, `catalog().applicability`, `float_basis` — rather
  than left in a docstring somewhere.
* **Freshness came from `coverage()`**, called before anything was claimed.

The contract these rules come from is
[Conventions & limits](https://octo-98895abd.mintlify.site/api-docs/conventions),
and its machine-readable twin is `POST /rpc/catalog`.